# 06 — Training and Generalization

This chapter explores how neural networks learn from training data and how to assess whether they generalize to unseen examples.

We will study training, validation, and test sets; underfitting and overfitting; learning curves; early stopping; weight decay; dropout; learning-rate selection; and learning-rate scheduling.

**Guiding principle:** training performance alone is not enough. We need a disciplined evaluation procedure and experiments that help explain model behaviour.

Run the notebook from top to bottom. Training cells may take time on CPU. The learning-rate comparison uses three epochs per setting to keep it short.

## 1. Imports and device selection

We prefer Intel XPU when available, then CUDA, then CPU.

In [ ]:
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.xpu.is_available():
    device = torch.device("xpu")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)

## 2. Generalization: training vs. validation vs. test

- **Training set:** used to calculate gradients and update parameters.
- **Validation set:** used during development to compare models and select hyperparameters or stopping points.
- **Test set:** reserved for final evaluation after development choices are complete.

The validation set does not directly update model weights. However, repeatedly making choices based on validation results can overfit the development process to the validation set.

CIFAR-10 has 50,000 images in its original training partition and 10,000 in its test partition. We split the original training partition into 40,000 training images and 10,000 validation images. The test set remains separate.

### Underfitting, generalization, and overfitting

| Pattern | Training performance | Validation performance |
|---|---|---|
| Underfitting | Poor | Poor |
| Good generalization | Good | Good |
| Overfitting | Very good | Noticeably worse |

A model with 99% training accuracy and 71% validation accuracy may have overfit. High training accuracy alone is not enough to conclude that a model is useful on unseen data.

## 3. Load CIFAR-10 and create the split

We use `ToTensor()` for a straightforward baseline. Augmentation is kept separate so its effects are not confused with regularization and learning-rate changes.

In [ ]:
DATA_ROOT = "/data/Datasets/Images/First_CNN_CIFAR10_Dataset/"
transform = transforms.ToTensor()

full_train_dataset = datasets.CIFAR10(
    root=DATA_ROOT, train=True, download=True, transform=transform
)
test_dataset = datasets.CIFAR10(
    root=DATA_ROOT, train=False, download=True, transform=transform
)

train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
split_generator = torch.Generator().manual_seed(SEED)

train_dataset, val_dataset = random_split(
    full_train_dataset, [train_size, val_size], generator=split_generator
)

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

classes = (
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
)

print(f"Training examples:   {len(train_dataset):,}")
print(f"Validation examples: {len(val_dataset):,}")
print(f"Test examples:       {len(test_dataset):,}")

## 4. Define the CNN

Two convolutional blocks reduce spatial dimensions from 32×32 to 16×16 and then 8×8. The final tensor has shape `64 × 8 × 8`, which flattens to 4096 features per image.

In [ ]:
class CNN(nn.Module):
    """Two-block convolutional neural network for CIFAR-10."""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(64 * 8 * 8, 10)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))  # [B, 32, 16, 16]
        x = self.pool(self.relu(self.conv2(x)))  # [B, 64, 8, 8]
        x = self.flatten(x)                      # [B, 4096]
        return self.fc(x)                        # [B, 10] logits

model = CNN().to(device)
print(model)

example_images, example_labels = next(iter(train_loader))
example_outputs = model(example_images[:8].to(device))
print("Input batch:", example_images[:8].shape)
print("Output batch:", example_outputs.shape)

## 5. Training and evaluation functions

`model.train()` enables training behaviour such as dropout. `model.eval()` switches modules to evaluation behaviour. `torch.no_grad()` avoids recording gradients during evaluation.

In [ ]:
def run_training_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        n = labels.size(0)
        total_loss += loss.item() * n
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += n

    return total_loss / total, correct / total


def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = loss_fn(logits, labels)

            n = labels.size(0)
            total_loss += loss.item() * n
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += n

    return total_loss / total, correct / total

## 6. Baseline experiment

Train the CNN for 20 epochs with Adam and learning rate `0.001`. We record training and validation loss and accuracy at every epoch.

In [ ]:
def fit_model(
    model, train_loader, val_loader, epochs=20, lr=0.001,
    weight_decay=0.0, use_early_stopping=False, patience=3, verbose=True
):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_loss = float("inf")
    best_epoch, best_state, patience_counter = 0, None, 0

    for epoch in range(epochs):
        train_loss, train_acc = run_training_epoch(
            model, train_loader, loss_fn, optimizer, device
        )
        val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        improved = val_loss < best_val_loss
        if improved:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if verbose:
            marker = " | New best validation loss" if improved else ""
            print(
                f"Epoch {epoch + 1:02d}/{epochs} | "
                f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | "
                f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%{marker}"
            )

        if use_early_stopping and patience_counter >= patience:
            if verbose:
                print(f"Early stopping at epoch {epoch + 1}.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    summary = {
        "best_epoch_by_val_loss": best_epoch,
        "best_val_loss": best_val_loss,
        "best_val_accuracy_seen": max(history["val_acc"]),
        "stopping_epoch": len(history["train_loss"]),
    }
    return history, summary

In [ ]:
baseline_model = CNN().to(device)
baseline_history, baseline_summary = fit_model(
    baseline_model, train_loader, val_loader,
    epochs=20, lr=0.001, weight_decay=0.0
)
print("\nBaseline summary:", baseline_summary)

### Plot the learning curves

A common overfitting pattern is training loss continuing to fall while validation loss stops improving or begins to rise. The exact curve varies between runs.

In [ ]:
def plot_history(history, title="Training history"):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history["train_loss"], label="Training loss")
    axes[0].plot(epochs, history["val_loss"], label="Validation loss")
    axes[0].set(xlabel="Epoch", ylabel="Loss", title="Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, np.array(history["train_acc"]) * 100, label="Training accuracy")
    axes[1].plot(epochs, np.array(history["val_acc"]) * 100, label="Validation accuracy")
    axes[1].set(xlabel="Epoch", ylabel="Accuracy (%)", title="Accuracy")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_history(baseline_history, "Baseline CNN")

## 7. Early stopping

Early stopping stops when the monitored validation loss fails to improve for `patience` consecutive epochs. The best checkpoint is restored.

- `best_epoch_by_val_loss` is the epoch with the lowest validation loss.
- `stopping_epoch` is when training ended.
- These are not necessarily the same.
- The epoch with the highest validation accuracy may differ from the lowest-loss epoch.

In [ ]:
early_stop_model = CNN().to(device)
early_history, early_summary = fit_model(
    early_stop_model, train_loader, val_loader,
    epochs=30, lr=0.001, weight_decay=0.0,
    use_early_stopping=True, patience=3
)
print("\nEarly-stopping summary:", early_summary)
plot_history(early_history, "CNN with early stopping")

## 8. Weight decay

Weight decay discourages unnecessarily large parameter values and may help reduce overfitting. Too much regularization can limit learning.

The exact implementation depends on the optimizer. AdamW uses decoupled weight decay, which is not identical to adding an L2 penalty directly to the loss.

In [ ]:
weight_decay_model = CNN().to(device)
wd_history, wd_summary = fit_model(
    weight_decay_model, train_loader, val_loader,
    epochs=20, lr=0.001, weight_decay=0.0001,
    use_early_stopping=True, patience=3
)
print("\nWeight-decay summary:", wd_summary)
plot_history(wd_history, "CNN with weight decay")

## 9. Dropout

Dropout randomly sets selected activations to zero during training. It does not permanently delete neurons.

- `model.train()` activates dropout.
- `model.eval()` disables dropout.
- PyTorch uses inverted-dropout scaling during training to preserve expected activation magnitude.
- Training accuracy can be lower than validation accuracy because dropout is active during training but disabled during evaluation.

In [ ]:
class CNNWithDropout(nn.Module):
    """CNN with dropout applied to flattened convolutional features."""
    def __init__(self, dropout_p=0.5):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p=dropout_p)
        self.fc = nn.Linear(64 * 8 * 8, 10)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.flatten(x)
        x = self.dropout(x)
        return self.fc(x)

dropout_model = CNNWithDropout(dropout_p=0.5).to(device)
dropout_history, dropout_summary = fit_model(
    dropout_model, train_loader, val_loader,
    epochs=30, lr=0.001, weight_decay=0.0001,
    use_early_stopping=True, patience=3
)
print("\nDropout + weight-decay summary:", dropout_summary)
plot_history(dropout_history, "CNN with dropout and weight decay")

### Observe dropout's train/evaluation behaviour

With dropout enabled, the same input can produce different outputs across training-mode forward passes. In evaluation mode, dropout is disabled, so repeated outputs are equal when the rest of the model is deterministic.

In [ ]:
sample_images, _ = next(iter(val_loader))
sample = sample_images[:1].to(device)

dropout_model.train()
with torch.no_grad():
    train_output_1 = dropout_model(sample)
    train_output_2 = dropout_model(sample)

dropout_model.eval()
with torch.no_grad():
    eval_output_1 = dropout_model(sample)
    eval_output_2 = dropout_model(sample)

print("Training-mode outputs equal:", torch.allclose(train_output_1, train_output_2))
print("Evaluation-mode outputs equal:", torch.allclose(eval_output_1, eval_output_2))

## 10. Learning rate

The learning rate controls the scale of optimizer updates. For gradient descent:

\[
w_{\text{new}} = w_{\text{old}} - \eta \frac{\partial L}{\partial w}
\]

A very small learning rate may make progress slowly. A very large rate may overshoot useful regions, cause unstable updates, or prevent convergence.

### Controlled comparison

We compare `0.00001`, `0.001`, and `0.1` for three epochs. Each experiment starts from the same model weights. Data shuffling and dropout can still introduce randomness, so this is a demonstration rather than a perfectly deterministic benchmark.

In [ ]:
torch.manual_seed(SEED)
base_model = CNNWithDropout(dropout_p=0.5).to(device)
initial_state = copy.deepcopy(base_model.state_dict())

learning_rates = [0.00001, 0.001, 0.1]
lr_results = {}

for lr in learning_rates:
    experiment_model = CNNWithDropout(dropout_p=0.5).to(device)
    experiment_model.load_state_dict(initial_state)

    history, summary = fit_model(
        experiment_model, train_loader, val_loader,
        epochs=3, lr=lr, weight_decay=0.0001,
        use_early_stopping=False, verbose=True
    )
    lr_results[lr] = {
        "history": history,
        "summary": summary,
        "final_train_loss": history["train_loss"][-1],
        "final_val_loss": history["val_loss"][-1],
    }

print("\nFinal comparison:")
for lr, result in lr_results.items():
    print(
        f"LR={lr:g}: Train Loss={result['final_train_loss']:.4f}, "
        f"Val Loss={result['final_val_loss']:.4f}"
    )

### Interpret the comparison

- A tiny learning rate may improve loss slowly.
- A suitable learning rate may make faster, steadier progress.
- A very large learning rate may cause overshooting, poor progress, or instability.

The best value depends on the model, optimizer, dataset, and schedule. A short run does not prove a globally optimal learning rate.

In [ ]:
plt.figure(figsize=(8, 5))
for lr, result in lr_results.items():
    epochs = range(1, len(result["history"]["val_loss"]) + 1)
    plt.plot(epochs, result["history"]["val_loss"], marker="o", label=f"lr={lr:g}")

plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("Learning-rate comparison")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 11. Learning-rate scheduling

A scheduler changes the learning rate during training. `StepLR` multiplies the learning rate by `gamma` every `step_size` scheduler steps. For an epoch-based schedule, call `scheduler.step()` once at the end of each epoch.

In [ ]:
scheduler_demo_model = CNN().to(device)
scheduler_optimizer = torch.optim.Adam(scheduler_demo_model.parameters(), lr=0.01)
scheduler = torch.optim.lr_scheduler.StepLR(
    scheduler_optimizer, step_size=5, gamma=0.5
)

schedule_values = []
for epoch in range(12):
    current_lr = scheduler_optimizer.param_groups[0]["lr"]
    schedule_values.append(current_lr)
    print(f"Epoch {epoch + 1:02d}: learning rate = {current_lr:.6f}")
    scheduler.step()

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(schedule_values) + 1), schedule_values, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Learning rate")
plt.title("StepLR schedule")
plt.grid(True, alpha=0.3)
plt.show()

## 12. Final test evaluation

Use the test set for final evaluation after model and hyperparameter choices are complete. Do not repeatedly tune against the test set.

This cell evaluates the dropout model from this notebook run; its result may differ from previous experiments.

In [ ]:
test_loss, test_acc = evaluate(
    dropout_model, test_loader, nn.CrossEntropyLoss(), device
)
print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_acc * 100:.2f}%")

## 13. Summary

| Technique | Main role |
|---|---|
| Early stopping | Controls when training ends |
| Weight decay | Regularizes parameter updates |
| Dropout | Randomly removes activations during training |
| Learning-rate scheduling | Changes update scale during training |

### Key takeaways

- Generalization matters more than training accuracy alone.
- Keep training, validation, and test roles separate.
- Use validation curves to diagnose possible overfitting.
- Restore the checkpoint with the best monitored validation metric.
- Weight decay and dropout regularize different parts of training.
- Learning-rate selection strongly affects optimization.
- Learning-rate scheduling can help refine learning over time.
- Compare experiments carefully and avoid universal conclusions from one run.

## Next chapter

**Sequence Models → RNNs → LSTMs/GRUs → Attention → Transformers**